# Feature Engineering

## Overview
This notebook demonstrates the exact preprocessing and feature generation pipeline used to build the final CatBoost model in production. The model relies heavily on historical trends, geographical clusters, and non-linear climate interactions.

We produce a final dataset of 60 variables.

## 1. Interaction Features
We calculate ratios between temperature and precipitation, humidity thresholds, and growing degree days (GDD).

In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import KFold
import warnings
warnings.filterwarnings('ignore')

RS = 42

df = pd.read_csv('../data/processed/final_data_for_web.csv')
df = df[np.isfinite(df['crop_yield']) & np.isfinite(df['area'])]
df = df[(df['crop_yield'] > 0) & (df['area'] > 0)]
df = df[df['crop_yield'] <= 500.0]

for col in ['season', 'crop', 'state_name', 'district_name']:
    df[col] = df[col].astype(str).str.lower().str.strip()

key_cols = ['state_name', 'district_name', 'season', 'crop', 'crop_year']
df = df.drop_duplicates(subset=key_cols, keep='first')

train_mask = df['crop_year'] <= 2017
for crop_name in df.loc[train_mask, 'crop'].unique():
    mask = train_mask & (df['crop'] == crop_name)
    vals = df.loc[mask, 'crop_yield']
    if len(vals) < 30: continue
    q1, q3 = vals.quantile(0.05), vals.quantile(0.95)
    iqr = q3 - q1
    lower, upper = max(0.001, q1 - 3.0 * iqr), q3 + 3.0 * iqr
    df.loc[mask, 'crop_yield'] = vals.clip(lower=lower, upper=upper)

df = df.sort_values(['crop_year', 'state_name', 'district_name']).reset_index(drop=True)
df['state_crop'] = df['state_name'] + '_' + df['crop']
df['crop_season'] = df['crop'] + '_' + df['season']
df['district_crop'] = df['district_name'] + '_' + df['crop']

df['log_area'] = np.log1p(df['area'])
df['log_precip'] = np.log1p(df['precipitation_sum'].clip(lower=0))

df['temp_precip_ratio'] = df['temperature_2m_mean'] / (df['precipitation_sum'] + 1.0)
df['precip_per_temp'] = df['precipitation_sum'] / (df['temperature_2m_mean'] + 1.0)
df['temp_x_humidity'] = df['temperature_2m_mean'] * df['relative_humidity_2m_mean'] / 100.0
df['precip_x_humidity'] = df['precipitation_sum'] * df['relative_humidity_2m_mean'] / 100.0
df['wind_x_temp'] = df['wind_speed_10m_mean'] * df['temperature_2m_mean'] / 100.0
df['humidity_per_wind'] = df['relative_humidity_2m_mean'] / (df['wind_speed_10m_mean'] + 1.0)
df['gdd_proxy'] = np.maximum(0, df['temperature_2m_mean'] - 10.0)
df['gdd_x_precip'] = df['gdd_proxy'] * df['log_precip']
df['temp_squared'] = df['temperature_2m_mean'] ** 2
df['precip_squared'] = df['precipitation_sum'] ** 2

df['is_drought'] = (df['precipitation_sum'] < 5.0).astype(int)
df['is_extreme_heat'] = (df['temperature_2m_mean'] > 35.0).astype(int)
df['is_high_humidity'] = (df['relative_humidity_2m_mean'] > 80.0).astype(int)

df['years_since_2001'] = df['crop_year'] - 2001
df['year_norm_sq'] = ((df['crop_year'] - 2001) / 19.0) ** 2
season_map = {'kharif': 0, 'rabi': 1, 'whole year': 2, 'summer': 3, 'winter': 4, 'autumn': 5}
season_ord = df['season'].map(season_map).fillna(2)
df['season_sin'] = np.sin(2 * np.pi * season_ord / 6)
df['season_cos'] = np.cos(2 * np.pi * season_ord / 6)


## 2. Temporal Lags and Trends
Historical yield is the strongest predictor of future yield. We use Expanding Means, Rolling Means, and Exponentially Weighted Moving Averages (EWMA) grouped by State, District, and Crop.

In [2]:

global_yield = df.loc[df['crop_year'] <= 2017, 'crop_yield'].mean()
def compute_lags(frame, group_col):
    grp = frame.groupby([group_col, 'crop_year'])['crop_yield'].mean().reset_index()
    grp[f'exp_mean_{group_col}'] = grp.groupby(group_col)['crop_yield'].transform(lambda x: x.expanding().mean().shift(1)).fillna(global_yield)
    grp[f'roll3_mean_{group_col}'] = grp.groupby(group_col)['crop_yield'].transform(lambda x: x.rolling(3, min_periods=1).mean().shift(1)).fillna(global_yield)
    grp[f'ewma_{group_col}'] = grp.groupby(group_col)['crop_yield'].transform(lambda x: x.ewm(alpha=0.3, adjust=False).mean().shift(1)).fillna(global_yield)
    merge_cols = [group_col, 'crop_year', f'exp_mean_{group_col}', f'roll3_mean_{group_col}', f'ewma_{group_col}']
    return frame.merge(grp[merge_cols], on=[group_col, 'crop_year'], how='left')

for col in ['state_crop', 'district_crop', 'crop', 'state_name']:
    df = compute_lags(df, col)
df['yield_trend_sc'] = df['exp_mean_state_crop'] - df['roll3_mean_state_crop']

scl = df.groupby(['state_name', 'crop_year'])[['temperature_2m_mean', 'precipitation_sum']].mean().reset_index()
scl['ht'] = scl.groupby('state_name')['temperature_2m_mean'].transform(lambda x: x.expanding().mean().shift(1)).fillna(scl['temperature_2m_mean'].mean())
scl['hp'] = scl.groupby('state_name')['precipitation_sum'].transform(lambda x: x.expanding().mean().shift(1)).fillna(scl['precipitation_sum'].mean())
df = df.merge(scl[['state_name', 'crop_year', 'ht', 'hp']], on=['state_name', 'crop_year'], how='left')
df['temp_anomaly'] = df['temperature_2m_mean'] - df['ht']
df['precip_anomaly'] = df['precipitation_sum'] - df['hp']
df.drop(columns=['ht', 'hp'], inplace=True)


## 3. Geospatial Clustering and PCA
We use K-Means on Latitude and Longitude to group geographical zones. We also apply Principal Component Analysis (PCA) to orthogonalize the climate variables.

In [3]:

train_mask = df['crop_year'] <= 2017.0
ll = df[['latitude', 'longitude']].values
for k in [10, 30, 50]:
    km = KMeans(n_clusters=k, random_state=RS, n_init=5)
    geo = np.zeros(len(df), dtype=int)
    geo[train_mask] = km.fit_predict(ll[train_mask])
    geo[~train_mask] = km.predict(ll[~train_mask])
    df[f'geo_cluster_{k}'] = geo

climate_cols = ['temperature_2m_mean', 'precipitation_sum', 'relative_humidity_2m_mean', 'wind_speed_10m_mean']
scaler = StandardScaler()
pca = PCA(n_components=2, random_state=RS)
c_train = scaler.fit_transform(df.loc[train_mask, climate_cols])
pca.fit(c_train)
pca_feats = pca.transform(scaler.transform(df[climate_cols]))
df['climate_pca_1'] = pca_feats[:, 0]
df['climate_pca_2'] = pca_feats[:, 1]

for col in ['district_name', 'state_name', 'crop', 'season']:
    freq = df.loc[train_mask, col].value_counts()
    df[f'freq_{col}'] = df[col].map(freq).fillna(0)

train_df = df[df['crop_year'] <= 2017].copy().reset_index(drop=True)
test_df = df[df['crop_year'] >= 2018].copy().reset_index(drop=True)

y_train_raw = train_df['crop_yield'].values.astype(np.float64)
y_test_raw = test_df['crop_yield'].values.astype(np.float64)
y_train = np.log1p(y_train_raw)


## 4. Target Encoding
Leave-One-Out Target Encoding (LOOE) is applied to high cardinality categorical columns to convert them to numeric yield baselines without leaking data from the current fold.

In [4]:

te_cols = ['crop', 'state_name', 'district_name', 'state_crop', 'district_crop']
kf = KFold(n_splits=5, shuffle=True, random_state=RS)
for col in te_cols:
    looe_col = f'looe_{col}'
    train_df[looe_col] = np.nan
    global_mean = y_train.mean()
    for tr_idx, val_idx in kf.split(train_df):
        X_tr, X_val = train_df.iloc[tr_idx], train_df.iloc[val_idx]
        stats = pd.Series(y_train[tr_idx]).groupby(X_tr[col].values).mean()
        train_df.loc[val_idx, looe_col] = X_val[col].map(stats)
    train_df[looe_col] = train_df[looe_col].fillna(global_mean)
    stats_full = pd.Series(y_train).groupby(train_df[col].values).mean()
    test_df[looe_col] = test_df[col].map(stats_full).fillna(global_mean)

LABEL_CATS = ['season', 'crop', 'state_name', 'geo_cluster_10', 'geo_cluster_30', 'geo_cluster_50']
for col in LABEL_CATS:
    le = LabelEncoder()
    le_col = f'{col}_le'
    train_df[le_col] = le.fit_transform(train_df[col].astype(str))
    known = set(le.classes_)
    test_df[le_col] = le.transform(test_df[col].astype(str).map(lambda x: x if x in known else le.classes_[0]))

CATBOOST_CAT = ['season', 'crop', 'state_name', 'crop_season', 'state_crop', 'geo_cluster_10', 'geo_cluster_30', 'geo_cluster_50']
for cdf in [train_df, test_df]:
    for c in CATBOOST_CAT:
        cdf[c] = cdf[c].astype(str)

NUM_FEATURES = [
    'crop_year', 'log_area', 'log_precip',
    'temperature_2m_mean', 'precipitation_sum', 'relative_humidity_2m_mean', 'wind_speed_10m_mean',
    'latitude', 'longitude',
    'temp_precip_ratio', 'precip_per_temp', 'temp_x_humidity', 'precip_x_humidity', 'wind_x_temp', 'humidity_per_wind',
    'is_drought', 'is_extreme_heat', 'is_high_humidity',
    'gdd_proxy', 'gdd_x_precip', 'temp_squared', 'precip_squared',
    'years_since_2001', 'year_norm_sq', 'season_sin', 'season_cos',
    'exp_mean_state_crop', 'roll3_mean_state_crop', 'ewma_state_crop',
    'exp_mean_district_crop', 'roll3_mean_district_crop', 'ewma_district_crop',
    'exp_mean_crop', 'roll3_mean_crop', 'ewma_crop',
    'exp_mean_state_name', 'roll3_mean_state_name', 'ewma_state_name',
    'yield_trend_sc', 'temp_anomaly', 'precip_anomaly',
    'climate_pca_1', 'climate_pca_2',
    'freq_district_name', 'freq_state_name', 'freq_crop', 'freq_season',
    'looe_crop', 'looe_state_name', 'looe_district_name', 'looe_state_crop', 'looe_district_crop'
]
CAT_FEATURES = NUM_FEATURES + CATBOOST_CAT

X_train_cat = train_df[CAT_FEATURES].fillna(0)
X_test_cat = test_df[CAT_FEATURES].fillna(0)

Path('../checkpoints').mkdir(exist_ok=True)
X_train_cat.to_parquet('../checkpoints/X_train_cat.parquet', index=False)
X_test_cat.to_parquet('../checkpoints/X_test_cat.parquet', index=False)
np.save('../checkpoints/y_train.npy', y_train)
np.save('../checkpoints/y_test.npy', y_test_raw)

print(f"Engineered {len(CAT_FEATURES)} features correctly.")


Engineered 60 features correctly.
